In [1]:
#Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#1. Create an input text file consisting of continuous, meaningful English text of approximately 10 pages. The text may be collected from books or articles and should be free of tables, code, or bullet points.
!pip install PyPDF2

import PyPDF2
import re

# Replace with your uploaded PDF name
pdf_file = "/content/drive/MyDrive/Colab Notebooks/Natural Language Processing/Document.pdf"

# Read PDF
text = ""
with open(pdf_file, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

# Optional cleaning:
# Remove extra spaces
text = re.sub(r'\s+', ' ', text)

# Remove bullet-like symbols if present
text = re.sub(r'•|\*|- ', '', text)

# Save as text file
with open("input_text.txt", "w", encoding="utf-8") as output:
    output.write(text)

print("PDF successfully converted to input_text.txt")
print("Approximate word count:", len(text.split()))

from google.colab import files
files.download("input_text.txt")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.3 MB/s eta 0:00:00
PDF successfully converted to input_text.txt
Approximate word count: 13033


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
#2. Load the input text and perform necessary preprocessing steps such as sentence segmentation, tokenization, and case folding.
!pip install nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Load your text file
with open("input_text.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Total Characters:", len(text))
print("Total Words (approx):", len(text.split()))

#Sentence Segmentation
from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(text)

print("Total Sentences:", len(sentences))
print("\nFirst 5 Sentences:\n")
for i in range(5):
    print(f"{i+1}. {sentences[i]}")

#Tokenization
from nltk.tokenize import word_tokenize

tokens = word_tokenize(text)

print("Total Tokens:", len(tokens))
print("\nFirst 20 Tokens:\n")
print(tokens[:20])

#Case Folding
tokens_lower = [word.lower() for word in tokens]

print("\nFirst 20 Tokens After Case Folding:\n")
print(tokens_lower[:20])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Total Characters: 79043
Total Words (approx): 13033
Total Sentences: 467

First 5 Sentences:

1. PREFACE TO THE EDINBURGH EDITION OF THE WORKS OF JOHN GALT John Galt was among the most popular and prolific Scottish writers of the nineteenth century.
2. He wrote in a panoply of forms and genres about a great variety of topics and settings, drawing on his experiences of living, working, and travelling in Scotland and England, in Europe and the Mediterranean, and in North America.
3. Y et only a fraction of his many works have been reprinted since their original publication.
4. In 1841–43 Galt’s most important publisher, Blackwood, reprinted seven of his novels in volumes 1, 2, 4, and 6 of the Blackwood’s Standard Novels series.
5. In 1895, the Blackwood firm republished these novels, with one change to the selection, as the eight-volume Works of John Galt ; this collection was reissued in 1936, again with one additional novel.
Total Tokens: 15217

First 20 Tokens:

['PREFACE', 'TO', 'THE

In [5]:
'''3. From the preprocessed text, compute the frequency counts for:
•	(a) Unigrams
•	(b) Bi-grams
•	(c) Tri-grams'''

from nltk.tokenize import word_tokenize
import re

# Load text
with open("input_text.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Tokenization
tokens = word_tokenize(text)

# Case folding
tokens = [word.lower() for word in tokens]

# Remove punctuation and non-alphabetic tokens
tokens = [word for word in tokens if word.isalpha()]

print("Total Clean Tokens:", len(tokens))

# (a) Unigram Frequency Count
from collections import Counter

unigram_counts = Counter(tokens)

print("Top 10 Unigrams:")
print(unigram_counts.most_common(10))

#(b) Bigram Frequency Count
from nltk.util import ngrams

bigrams = list(ngrams(tokens, 2))
bigram_counts = Counter(bigrams)

print("\nTop 10 Bigrams:")
print(bigram_counts.most_common(10))

#(c) Trigram Frequency Count
trigrams = list(ngrams(tokens, 3))
trigram_counts = Counter(trigrams)

print("\nTop 10 Trigrams:")
print(trigram_counts.most_common(10))

#Overall
print("----- SUMMARY -----")
print("Total Unique Unigrams:", len(unigram_counts))
print("Total Unique Bigrams:", len(bigram_counts))
print("Total Unique Trigrams:", len(trigram_counts))

Total Clean Tokens: 12306
Top 10 Unigrams:
[('the', 962), ('of', 647), ('and', 366), ('in', 358), ('to', 313), ('a', 272), ('galt', 193), ('s', 169), ('his', 142), ('that', 141)]

Top 10 Bigrams:
[(('of', 'the'), 192), (('in', 'the'), 101), (('galt', 's'), 68), (('the', 'omen'), 57), (('andrew', 'of'), 48), (('of', 'padua'), 48), (('to', 'the'), 40), (('and', 'the'), 40), (('of', 'his'), 26), (('to', 'be'), 25)]

Top 10 Trigrams:
[(('andrew', 'of', 'padua'), 48), (('of', 'galt', 's'), 20), (('of', 'the', 'omen'), 19), (('s', 'edinburgh', 'magazine'), 15), (('to', 'william', 'blackwood'), 14), (('of', 'john', 'galt'), 13), (('blackwood', 's', 'edinburgh'), 11), (('of', 'padua', 'the'), 11), (('the', 'end', 'of'), 10), (('as', 'well', 'as'), 9)]
----- SUMMARY -----
Total Unique Unigrams: 2884
Total Unique Bigrams: 8755
Total Unique Trigrams: 11288


In [7]:
'''4. Using the computed counts, estimate the probability distributions for:
•	(a) Unigram language model
•	(b) Bi-gram language model
•	(c) Tri-gram language model
Clearly specify the probability equations used for each model.'''

#(a) Unigram Language Model
# Total number of tokens
N = len(tokens)

# Compute unigram probabilities
unigram_prob = {word: count / N for word, count in unigram_counts.items()}

# Show top 10 unigram probabilities
print("Top 10 Unigram Probabilities:")
for word, prob in list(unigram_prob.items())[:10]:
    print(f"{word} : {prob:.6f}")

#(b) Bi-gram language model
# Compute bigram probabilities
bigram_prob = {}

for (w1, w2), count in bigram_counts.items():
    bigram_prob[(w1, w2)] = count / unigram_counts[w1]

# Show top 10 bigram probabilities
print("\nTop 10 Bigram Probabilities:")
for bigram, prob in list(bigram_prob.items())[:10]:
    print(f"{bigram} : {prob:.6f}")

#(c)Tri-gram language model
# Compute trigram probabilities
trigram_prob = {}

for (w1, w2, w3), count in trigram_counts.items():
    trigram_prob[(w1, w2, w3)] = count / bigram_counts[(w1, w2)]

# Show top 10 trigram probabilities
print("\nTop 10 Trigram Probabilities:")
for trigram, prob in list(trigram_prob.items())[:10]:
    print(f"{trigram} : {prob:.6f}")


Top 10 Unigram Probabilities:
preface : 0.000406
to : 0.025435
the : 0.078173
edinburgh : 0.003982
edition : 0.001138
of : 0.052576
works : 0.001544
john : 0.002600
galt : 0.015683
was : 0.006338

Top 10 Bigram Probabilities:
('preface', 'to') : 0.200000
('to', 'the') : 0.127796
('the', 'edinburgh') : 0.009356
('edinburgh', 'edition') : 0.102041
('edition', 'of') : 0.500000
('of', 'the') : 0.296754
('the', 'works') : 0.007277
('works', 'of') : 0.421053
('of', 'john') : 0.020093
('john', 'galt') : 0.593750

Top 10 Trigram Probabilities:
('preface', 'to', 'the') : 1.000000
('to', 'the', 'edinburgh') : 0.025000
('the', 'edinburgh', 'edition') : 0.555556
('edinburgh', 'edition', 'of') : 1.000000
('edition', 'of', 'the') : 1.000000
('of', 'the', 'works') : 0.031250
('the', 'works', 'of') : 1.000000
('works', 'of', 'john') : 0.875000
('of', 'john', 'galt') : 1.000000
('john', 'galt', 'john') : 0.105263


In [9]:
'''5. Using each of the trained models separately, generate a paragraph consisting of 5 meaningful sentences using:
•	(a) Unigram model
•	(b) Bi-gram model
•	(c) Tri-gram model'''

#(a) Generate Paragraph Using Unigram Model
import random

def generate_unigram_sentence(length=12):
    words = list(unigram_prob.keys())
    probabilities = list(unigram_prob.values())

    sentence = random.choices(words, weights=probabilities, k=length)
    return " ".join(sentence).capitalize() + "."

# Generate 5 sentences
print("---- Unigram Model Output ----\n")
for _ in range(5):
    print(generate_unigram_sentence())

#(b) Generate Paragraph Using Bigram Model
def generate_bigram_sentence(length=12):
    # Start with a random word
    current_word = random.choice(list(unigram_counts.keys()))
    sentence = [current_word]

    for _ in range(length - 1):
        candidates = [(w2, prob) for (w1, w2), prob in bigram_prob.items() if w1 == current_word]

        if not candidates:
            break

        words, probs = zip(*candidates)
        next_word = random.choices(words, weights=probs)[0]

        sentence.append(next_word)
        current_word = next_word

    return " ".join(sentence).capitalize() + "."

# Generate 5 sentences
print("\n---- Bigram Model Output ----\n")
for _ in range(5):
    print(generate_bigram_sentence())

#(c)Generate Paragraph Using Trigram Model
def generate_trigram_sentence(length=12):
    # Start with random bigram
    start_bigram = random.choice(list(bigram_counts.keys()))
    sentence = [start_bigram[0], start_bigram[1]]

    for _ in range(length - 2):
        w1, w2 = sentence[-2], sentence[-1]

        candidates = [(w3, prob) for (x1, x2, w3), prob in trigram_prob.items() if x1 == w1 and x2 == w2]

        if not candidates:
            break

        words, probs = zip(*candidates)
        next_word = random.choices(words, weights=probs)[0]

        sentence.append(next_word)

    return " ".join(sentence).capitalize() + "."

# Generate 5 sentences
print("\n---- Trigram Model Output ----\n")
for _ in range(5):
    print(generate_trigram_sentence())


---- Unigram Model Output ----

Contain consisted which february the for miss gentleman january to of romance.
Omen even to comedy of continued a and novellettes the effort writing.
Frustrations the claims well close contributes even in ever his library satire.
Theatre item a clark very but the approval literary evoke significant oldest.
Was this think change on and up a are thomas longer roman.

---- Bigram Model Output ----

Leads on contempo rary reading and his first trip to embody presentiments.
Introductionxxxvii page of the works the years and commercial negotiations rather than.
Revealed to refer to their own writing and it is to king.
Letters were resigned to galt s events to publish his autobiography of.
Vidin for the sake of padua in english nobleman named henry corresponds.

---- Trigram Model Output ----

Miss edgeworth paris persan pp introductionxxi to have declined the stories but.
The publisher phillips and souter also influenced the composition of glenfell who.
And ca

6. Compare the grammatical correctness and contextual coherence of the text generated by the three models.

The Unigram model fails to produce grammatically correct or meaningful text because it ignores context.

The Bigram model improves grammatical structure slightly by considering one previous word.

The Trigram model produces the most coherent and fluent text because it considers two previous words, capturing better contextual dependencies.

However, none of the models fully ensure complete grammatical correctness due to limited context and absence of deep syntactic understanding.

Increasing n-gram size improves fluency but also increases sparsity issues.

In [10]:
#7. Split the dataset into training and test portions.

# Define split ratio
split_ratio = 0.8

# Compute split index
split_index = int(len(tokens) * split_ratio)

# Sequential split
train_tokens = tokens[:split_index]
test_tokens = tokens[split_index:]

print("Total Tokens:", len(tokens))
print("Training Tokens:", len(train_tokens))
print("Test Tokens:", len(test_tokens))


Total Tokens: 12306
Training Tokens: 9844
Test Tokens: 2462


In [14]:
#8. Evaluate the perplexity of the unigram, bi-gram, and tri-gram models on the test data.

#1: Rebuild Models Using Training Data
from collections import Counter
from nltk.util import ngrams
import math

# --- Unigram Counts (Training Only) ---
train_unigram_counts = Counter(train_tokens)
train_total_tokens = len(train_tokens)

# --- Bigram Counts ---
train_bigram_counts = Counter(ngrams(train_tokens, 2))

# --- Trigram Counts ---
train_trigram_counts = Counter(ngrams(train_tokens, 3))

#(a) Unigram Perplexity
# Vocabulary size
V = len(train_unigram_counts)

log_prob = 0
N_test = len(test_tokens)

for word in test_tokens:
    count = train_unigram_counts[word]
    prob = (count + 1) / (train_total_tokens + V)
    log_prob += math.log(prob)

unigram_perplexity = math.exp(-log_prob / N_test)

print("Unigram Perplexity:", unigram_perplexity)

#(b) Bigram Perplexity
log_prob = 0
N_test = len(test_tokens)

for i in range(1, N_test):
    w1 = test_tokens[i-1]
    w2 = test_tokens[i]

    bigram_count = train_bigram_counts[(w1, w2)]
    unigram_count = train_unigram_counts[w1]

    prob = (bigram_count + 1) / (unigram_count + V)
    log_prob += math.log(prob)

bigram_perplexity = math.exp(-log_prob / (N_test - 1))

print("Bigram Perplexity:", bigram_perplexity)

#(c) Trigram Perplexity
log_prob = 0
N_test = len(test_tokens)

for i in range(2, N_test):
    w1 = test_tokens[i-2]
    w2 = test_tokens[i-1]
    w3 = test_tokens[i]

    trigram_count = train_trigram_counts[(w1, w2, w3)]
    bigram_count = train_bigram_counts[(w1, w2)]

    prob = (trigram_count + 1) / (bigram_count + V)
    log_prob += math.log(prob)

trigram_perplexity = math.exp(-log_prob / (N_test - 2))

print("Trigram Perplexity:", trigram_perplexity)

Unigram Perplexity: 637.7358272112698
Bigram Perplexity: 1405.9680536988574
Trigram Perplexity: 2277.3464336507677


9. Analyze the perplexity values obtained and explain how increasing the order of the n-gram model affects language modeling performance.

The Unigram model achieved the lowest perplexity, indicating better generalization on limited data.

The Bigram and Trigram models showed higher perplexity, due to data sparsity and insufficient training data.

Although higher-order models generated more grammatically coherent sentences (Q6), they performed worse quantitatively.

This demonstrates that increasing n improves contextual modeling but requires significantly larger datasets to outperform simpler models.